# ISL Engine — Colab pipeline v2 (resumable)
Everything important is saved on **Google Drive**, so a closed laptop, a disconnect or a new day only costs you the step that was running.

| Drive folder / file | Made by | If it exists |
|---|---|---|
| `INCLUDE50/` | Stage 1 | Stage 1 skips finished files |
| `isl_cache/` | Stage 3 | Stage 3 skips finished videos |
| `isl_runs/exp1/best.pt` | Stage 4 | you can go straight to Stage 5 |

**Every new session:** run **Setup** (and Smoke test), then continue with the first stage that is not finished. Finished stages finish fast. Nothing here deletes your data, so even *Run all* is safe (just slow).

Needed in My Drive: `isl-engine.zip`; later `custom.zip` (your `idle` folder zipped).
Stages 1–3 need no GPU (use a **CPU** runtime to save units). Before Stage 4 switch to **T4 GPU**.

## Setup — run at the start of EVERY session

In [ ]:
import os, sys
from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'):
    drive.mount('/content/drive', force_remount=True)
sh = get_ipython().system
sh('unzip -q -o /content/drive/MyDrive/isl-engine.zip -d /content')      # code only; never touches your data
os.chdir('/content/isl-engine'); sys.path.insert(0, '/content/isl-engine')
sh('pip install -q mediapipe datasets onnx onnxruntime')

# newest downloader (supports --keep, so only the INCLUDE-50 videos are stored)
DL = r'''import argparse, json, os, time, urllib.request, zipfile

API = os.getenv("ISL_ZENODO_API", "https://zenodo.org/api/records/4010759")


def fetch(url, dst, size, tries=12):
    """Download with automatic retry + resume (HTTP Range). Returns True on success."""
    part = dst + ".part"
    for t in range(tries):
        have = os.path.getsize(part) if os.path.exists(part) else 0
        if size and have == size:
            os.replace(part, dst); return True
        if size and have > size:
            os.remove(part); have = 0
        req = urllib.request.Request(url, headers={"Range": f"bytes={have}-"} if have else {})
        try:
            with urllib.request.urlopen(req, timeout=60) as r:
                mode = "ab" if (have and r.status == 206) else "wb"
                with open(part, mode) as f:
                    while True:
                        chunk = r.read(1 << 20)
                        if not chunk:
                            break
                        f.write(chunk)
        except Exception as e:
            print(f"  network problem ({type(e).__name__}); retry {t + 1}/{tries}", flush=True)
            time.sleep(min(60, 5 * (t + 1))); continue
        if not size or os.path.getsize(part) == size:
            os.replace(part, dst); return True
        print(f"  incomplete ({os.path.getsize(part)}/{size} bytes); resuming", flush=True)
    return False


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--list", action="store_true"); ap.add_argument("--all", action="store_true")
    ap.add_argument("--only", nargs="*", default=[]); ap.add_argument("--out", default="data/INCLUDE")
    ap.add_argument("--delete-zips", action="store_true"); ap.add_argument("--keep")
    a = ap.parse_args()
    keep = set(open(a.keep).read().split("\n")) if a.keep else None
    key = lambda n: "/".join(n.replace("\\", "/").split("/")[-3:]).lower()
    with urllib.request.urlopen(API) as r:
        files = [(f["key"], f["links"]["self"], f.get("size", 0)) for f in json.load(r)["files"]]
    for k, _, s in files:
        print(f"{s / 1e9:6.2f} GB  {k}")
    print(f"total {sum(s for *_, s in files) / 1e9:.1f} GB")
    if a.list or not (a.all or a.only):
        raise SystemExit
    os.makedirs(a.out, exist_ok=True)
    failed, total = [], 0
    for k, url, size in files:
        if not (a.all or any(o.lower() in k.lower() for o in a.only)):
            continue
        dst = os.path.join(a.out, k); done = dst + ".done"
        if os.path.exists(done):
            print("already done", k); continue
        if not os.path.exists(dst) or (size and os.path.getsize(dst) != size):
            print("downloading", k, flush=True)
            if not fetch(url, dst, size):
                print("  GAVE UP on", k, "- rerun this cell later", flush=True); failed.append(k); continue
        if k.lower().endswith(".zip"):
            print("unzipping", k, flush=True)
            n = 0
            with zipfile.ZipFile(dst) as z:
                for m in z.infolist():
                    if keep is None or key(m.filename) in keep:
                        z.extract(m, a.out); n += 1
            print("  extracted", n, "files", flush=True); total += n
        open(done, "w").close()
        if a.delete_zips and k.lower().endswith(".zip"):
            os.remove(dst)
    if failed:
        print("NOT FINISHED:", failed, "-> RERUN THIS CELL")
    else:
        print("ALL FINISHED ->", a.out, "(videos extracted this run:", total, ")")
'''
open('isl/download_include.py', 'w').write(DL)

# list of the ~943 INCLUDE-50 videos
from datasets import load_dataset
ds = load_dataset("ai4bharat/INCLUDE")
key = lambda p: "/".join(p.replace("\\", "/").split("/")[-3:]).lower()
keep = {key(r["video_path"]) for s in ds for r in ds[s] if r["include_50"]}
os.makedirs('data', exist_ok=True)
open('data/keep.txt', 'w').write("\n".join(sorted(keep)))
print(len(keep), "INCLUDE-50 videos listed. SETUP DONE")
for f in ('isl-engine.zip', 'custom.zip'):
    print(f, 'on Drive:', os.path.exists('/content/drive/MyDrive/' + f))

## Stage 1 — Download INCLUDE-50 videos into Drive (slow, resumable)
It retries and resumes by itself after network drops. Wait for the last line: `ALL FINISHED`. If it says `NOT FINISHED ... RERUN THIS CELL`, just run the cell again.

In [ ]:
%cd /content/isl-engine
!python -m isl.download_include --all --delete-zips --keep data/keep.txt --out /content/drive/MyDrive/INCLUDE50

## Stage 1 — Download INCLUDE-50 videos into Drive (slow, resumable)
Stops/disconnects are fine: just rerun this cell. Wait for the last line `done -> ...`.

In [ ]:
%cd /content/isl-engine
!python -m isl.download_include --all --delete-zips --keep data/keep.txt --out /content/drive/MyDrive/INCLUDE50

## Stage 2 — Build the training list
Needs `custom.zip` (idle clips) on Drive. Expect **51 classes**, a small 'missing on disk' number, and train/val/test counts.

In [ ]:
import os
sh = get_ipython().system
os.chdir('/content/isl-engine')
sh('mkdir -p data/custom; if [ -f /content/drive/MyDrive/custom.zip ]; then unzip -q -o /content/drive/MyDrive/custom.zip -d data/custom; fi')
idle = os.listdir('data/custom/idle') if os.path.isdir('data/custom/idle') else []
print(len(idle), 'idle clips found')
if len(idle) < 3:
    print("STOP: record idle clips on your PC, zip the 'idle' folder as custom.zip, upload it to My Drive, then rerun this cell.")
else:
    sh('python -m isl.hf_manifest --root /content/drive/MyDrive/INCLUDE50 --custom data/custom')

## Stage 3 — Extract hand/body landmarks (slow, resumable, saved to Drive)
Rerun after any interruption; finished videos are skipped.

In [ ]:
%cd /content/isl-engine
!python -m isl.extract --manifest data/manifest.csv --out /content/drive/MyDrive/isl_cache --workers 2

## Stage 4 — Train
First switch **Runtime → Change runtime type → T4 GPU** (this restarts Colab — fine, your data is on Drive), then run **Setup** again, then this cell. The first line must say `cuda`.

In [ ]:
%cd /content/isl-engine
!mkdir -p /content/drive/MyDrive/isl_runs
!python -m isl.train --index /content/drive/MyDrive/isl_cache/index.csv --out /content/drive/MyDrive/isl_runs/exp1

## Stage 5 — Export the model and download it
Put both files into `D:\PROJECTS\isl-engine\models\` on your PC.

In [ ]:
%cd /content/isl-engine
!mkdir -p models
!python -m isl.export_onnx --ckpt /content/drive/MyDrive/isl_runs/exp1/best.pt --out models/isl.onnx
!cp models/isl.onnx models/isl.labels.json /content/drive/MyDrive/
from google.colab import files
files.download('models/isl.onnx'); files.download('models/isl.labels.json')